In [ ]:
"""
Batch DOCX → PDF Converter
===========================
Inspired by: github.com/SakibAhmedShuva/batch-excel-xlsx-csv-to-pdf-converter

Converts .docx (and optionally legacy .doc) files to PDF in bulk.

Conversion backends (auto-detected, first available wins):
  1. LibreOffice  – searches common install locations on Windows / Mac / Linux
  2. mammoth + weasyprint  – pure Python fallback
     Install:  pip install mammoth weasyprint pypdf

Usage:
  1. Set INPUT_DIR to your folder of Word documents
  2. Run:  python batch_docx_to_pdf.py
  3. PDFs appear in OUTPUT_PDF_DIR (default: "output_pdf_files/")
"""

import os
import glob
import shutil
import socket
import subprocess
import sys
import tempfile
from datetime import datetime
from pathlib import Path

# ─────────────────────────────────────────────────────────────
# MAIN CONFIGURATION  ← edit these as needed
# Paths are relative to THIS script's folder, not wherever you
# run Python from. Or use absolute paths, e.g.:
#   INPUT_DIR     = r"C:\Users\You\Documents\my_docs"
#   OUTPUT_PDF_DIR = r"C:\Users\You\Documents\output"
# ─────────────────────────────────────────────────────────────
INPUT_DIR     = r"C:\Users\Public\Desktop\HP\New folder"
OUTPUT_PDF_DIR = r"C:\Users\Sakib\Desktop\output_pdf"

RECURSIVE_SEARCH = True
CREATE_SUBFOLDERS_IN_OUTPUT = True

FILE_EXTENSIONS_TO_CONVERT = [".docx", ".doc"]

OVERWRITE_EXISTING = True

# ── Merge option ─────────────────────────────────────────────
MERGE_ALL_INTO_ONE_PDF = False
MERGED_PDF_FILENAME = "merged_output.pdf"
# ─────────────────────────────────────────────────────────────


# ── logging ──────────────────────────────────────────────────

def _log(msg: str, level: str = "INFO") -> None:
    ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    prefix = {"ERROR": "✗  ", "SKIP": "–  ", "OK": "✓  "}.get(level, "   ")
    print(f"[{ts}] [{level}] {prefix}{msg}")


# ── LibreOffice path detection ────────────────────────────────

# Well-known install locations per platform
_LO_CANDIDATES = [
    # ← Your install path (takes priority)
    r"d:\Program Files\LibreOffice\program\soffice.exe",
    # Linux
    "soffice",
    "/usr/bin/soffice",
    "/usr/lib/libreoffice/program/soffice",
    "/snap/bin/soffice",
    # macOS
    "/Applications/LibreOffice.app/Contents/MacOS/soffice",
    "/Applications/LibreOffice.app/Contents/MacOS/LibreOffice",
    # Windows – expand env vars so they resolve at runtime
    r"%PROGRAMFILES%\LibreOffice\program\soffice.exe",
    r"%PROGRAMFILES(X86)%\LibreOffice\program\soffice.exe",
    r"%LOCALAPPDATA%\Programs\LibreOffice\program\soffice.exe",
    r"C:\Program Files\LibreOffice\program\soffice.exe",
    r"C:\Program Files (x86)\LibreOffice\program\soffice.exe",
]


def _find_soffice() -> str | None:
    """Return the full path to soffice/LibreOffice, or None."""
    # 1. shutil.which covers PATH + PATHEXT on all platforms
    found = shutil.which("soffice") or shutil.which("soffice.exe") or shutil.which("LibreOffice")
    if found:
        return found

    # 2. Walk the candidate list (expand env vars for Windows paths)
    for candidate in _LO_CANDIDATES:
        expanded = os.path.expandvars(candidate)
        if os.path.isfile(expanded):
            return expanded

    # 3. On Windows, also try to find via registry-style glob
    if sys.platform == "win32":
        for pattern in [
            r"C:\Program Files\LibreOffice*\program\soffice.exe",
            r"C:\Program Files (x86)\LibreOffice*\program\soffice.exe",
        ]:
            hits = glob.glob(pattern)
            if hits:
                return hits[0]

    return None


# ── backend: LibreOffice ──────────────────────────────────────

def _soffice_env() -> dict:
    env = os.environ.copy()
    if sys.platform.startswith("linux"):
        env["SAL_USE_VCLPLUGIN"] = "svp"
        needs_shim = False
        try:
            s = socket.socket(socket.AF_UNIX, socket.SOCK_STREAM)
            with tempfile.TemporaryDirectory() as tmp:
                s.bind(os.path.join(tmp, "t.sock"))
            s.close()
        except (OSError, AttributeError):
            needs_shim = True
        if needs_shim:
            shim_path = Path(tempfile.gettempdir()) / "no_unix_socket.so"
            if not shim_path.exists():
                src_c = shim_path.with_suffix(".c")
                src_c.write_text(
                    "#include <sys/socket.h>\n#include <errno.h>\n"
                    "int bind(int fd,const struct sockaddr*a,socklen_t l){"
                    "if(a&&a->sa_family==AF_UNIX){errno=ENOTSUP;return -1;}return 0;}\n"
                )
                subprocess.run(
                    ["gcc", "-shared", "-fPIC", "-o", str(shim_path), str(src_c)],
                    check=False, capture_output=True,
                )
            if shim_path.exists():
                env["LD_PRELOAD"] = str(shim_path)
    return env


def _convert_libreoffice(src: str, dst: str, soffice_bin: str) -> bool:
    # Convert into a temp dir to avoid src==dst dir conflicts and spaces issues
    with tempfile.TemporaryDirectory() as tmp_out:
        env = _soffice_env()
        try:
            r = subprocess.run(
                [soffice_bin, "--headless", "--convert-to", "pdf",
                 "--outdir", tmp_out, src],
                env=env, capture_output=True, text=True, timeout=120,
            )
        except subprocess.TimeoutExpired:
            _log(f"LibreOffice timed out on '{src}'", "ERROR")
            return False
        except Exception as e:
            _log(f"LibreOffice error: {e}", "ERROR")
            return False

        if r.returncode != 0:
            _log(f"LibreOffice failed (exit {r.returncode}): {r.stderr.strip() or r.stdout.strip()}", "ERROR")
            return False

        # Find whatever PDF LibreOffice produced in tmp_out
        produced = [f for f in os.listdir(tmp_out) if f.lower().endswith(".pdf")]
        if not produced:
            _log(f"LibreOffice ran but produced no PDF. stdout: {r.stdout.strip()}", "ERROR")
            return False

        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.move(os.path.join(tmp_out, produced[0]), dst)
        return os.path.exists(dst)


# ── backend: mammoth + weasyprint ────────────────────────────

_MAMMOTH_CSS = """
@page { size: A4; margin: 2cm; }
body  { font-family: Arial, Helvetica, sans-serif; font-size: 10pt; line-height: 1.45; color: #111; }
h1   { font-size: 18pt; margin: 0 0 12pt; }
h2   { font-size: 14pt; margin: 14pt 0 6pt; }
h3   { font-size: 12pt; margin: 10pt 0 4pt; }
p    { margin: 0 0 8pt; }
table { border-collapse: collapse; width: 100%; margin-bottom: 10pt; }
th, td { border: 1px solid #bbb; padding: 3px 7px; font-size: 9pt; }
th   { background: #f0f0f0; font-weight: bold; }
ul, ol { margin: 0 0 8pt 1.5em; padding: 0; }
li   { margin-bottom: 3pt; }
"""

def _convert_mammoth(src: str, dst: str) -> bool:
    try:
        import mammoth
        from weasyprint import HTML, CSS
    except ImportError:
        _log("mammoth / weasyprint not installed. Run: pip install mammoth weasyprint", "ERROR")
        return False
    try:
        with open(src, "rb") as f:
            result = mammoth.convert_to_html(f)
        html = (
            "<html><head><meta charset='UTF-8'></head>"
            f"<body>{result.value}</body></html>"
        )
        HTML(string=html).write_pdf(dst, stylesheets=[CSS(string=_MAMMOTH_CSS)])
        return os.path.exists(dst)
    except Exception as e:
        _log(f"mammoth/weasyprint error: {e}", "ERROR")
        return False


# ── backend selector ─────────────────────────────────────────

def _detect_backend():
    """Returns ("libreoffice", path) or ("mammoth", None) or ("none", None)."""
    soffice = _find_soffice()
    if soffice:
        return "libreoffice", soffice
    try:
        import mammoth       # noqa: F401
        import weasyprint    # noqa: F401
        return "mammoth", None
    except ImportError:
        pass
    return "none", None


def convert_to_pdf(src: str, dst: str, backend: str, soffice_bin: str | None) -> bool:
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    if backend == "libreoffice":
        return _convert_libreoffice(src, dst, soffice_bin)
    elif backend == "mammoth":
        return _convert_mammoth(src, dst)
    return False


# ── file discovery & path helpers ────────────────────────────

def collect_files(input_dir: str, extensions: list, recursive: bool) -> list:
    found = []
    exts = [e.lower() for e in extensions]
    if recursive:
        for root, _, files in os.walk(input_dir):
            for f in files:
                if any(f.lower().endswith(e) for e in exts):
                    found.append(os.path.join(root, f))
    else:
        for e in exts:
            found.extend(glob.glob(os.path.join(input_dir, f"*{e}")))
    return sorted(found)


def resolve_output_path(src: str, input_base: str, output_base: str, mirror: bool) -> str:
    stem = Path(src).stem
    if mirror:
        rel = os.path.relpath(os.path.dirname(src), input_base)
        out_dir = output_base if rel == "." else os.path.join(output_base, rel)
    else:
        out_dir = output_base
    os.makedirs(out_dir, exist_ok=True)
    return os.path.join(out_dir, stem + ".pdf")


# ── merge helper ─────────────────────────────────────────────

def merge_pdfs(pdf_paths: list, output_path: str) -> bool:
    try:
        from pypdf import PdfWriter
    except ImportError:
        _log("pypdf not installed. Run: pip install pypdf", "ERROR")
        return False
    writer = PdfWriter()
    for p in pdf_paths:
        try:
            writer.append(p)
        except Exception as e:
            _log(f"Could not include '{p}' in merge: {e}", "ERROR")
    if len(writer.pages) == 0:
        _log("No pages to merge.", "ERROR")
        return False
    with open(output_path, "wb") as f:
        writer.write(f)
    return os.path.exists(output_path)


# ── main ─────────────────────────────────────────────────────

def main() -> None:
    backend, soffice_bin = _detect_backend()

    if backend == "none":
        _log("No conversion backend found.", "ERROR")
        _log("")
        _log("Option A – LibreOffice (best quality):")
        _log("  Windows : https://www.libreoffice.org/download")
        _log("  Mac     : https://www.libreoffice.org/download")
        _log("  Linux   : sudo apt install libreoffice  OR  sudo dnf install libreoffice")
        _log("")
        _log("  If already installed, LibreOffice may not be on PATH.")
        _log("  Set SOFFICE_PATH at the top of this script to your soffice location, e.g.:")
        _log(r'  Windows: C:\Program Files\LibreOffice\program\soffice.exe')
        _log("  Mac    : /Applications/LibreOffice.app/Contents/MacOS/soffice")
        _log("")
        _log("Option B – Pure Python fallback:")
        _log("  pip install mammoth weasyprint pypdf")
        sys.exit(1)

    if not os.path.isdir(INPUT_DIR):
        _log(f"Input directory '{INPUT_DIR}' not found. Create it and add Word documents.", "ERROR")
        sys.exit(1)

    os.makedirs(OUTPUT_PDF_DIR, exist_ok=True)

    _log("=" * 58)
    _log("Batch DOCX → PDF Converter")
    _log(f"Backend  : {backend}" + (f"  ({soffice_bin})" if soffice_bin else ""))
    _log(f"Input    : {os.path.abspath(INPUT_DIR)}")
    _log(f"Output   : {os.path.abspath(OUTPUT_PDF_DIR)}")
    _log(f"Types    : {', '.join(FILE_EXTENSIONS_TO_CONVERT)}")
    _log(f"Recursive: {RECURSIVE_SEARCH}  |  Mirror folders: {CREATE_SUBFOLDERS_IN_OUTPUT}")
    _log(f"Merge all: {MERGE_ALL_INTO_ONE_PDF}")
    _log("-" * 58)

    files = collect_files(INPUT_DIR, FILE_EXTENSIONS_TO_CONVERT, RECURSIVE_SEARCH)
    if not files:
        _log("No files found. Check INPUT_DIR and FILE_EXTENSIONS_TO_CONVERT.")
        return

    _log(f"Found {len(files)} file(s) to process.\n")

    success_paths = []
    skip_count = fail_count = 0
    mirror = RECURSIVE_SEARCH and CREATE_SUBFOLDERS_IN_OUTPUT

    for idx, src in enumerate(files, 1):
        dst = resolve_output_path(src, INPUT_DIR, OUTPUT_PDF_DIR, mirror)
        _log(f"[{idx}/{len(files)}] {src}")

        if not OVERWRITE_EXISTING and os.path.exists(dst):
            _log(f"Skipping (already exists): {dst}", "SKIP")
            skip_count += 1
            success_paths.append(dst)
            print()
            continue

        ok = convert_to_pdf(src, dst, backend, soffice_bin)
        if ok:
            size_kb = os.path.getsize(dst) / 1024
            _log(f"Saved: {dst}  ({size_kb:.1f} KB)", "OK")
            success_paths.append(dst)
        else:
            fail_count += 1
        print()

    # ── optional merge ────────────────────────────────────────
    if MERGE_ALL_INTO_ONE_PDF and success_paths:
        merged_path = os.path.join(OUTPUT_PDF_DIR, MERGED_PDF_FILENAME)
        _log(f"Merging {len(success_paths)} PDF(s) → {merged_path}")
        if merge_pdfs(success_paths, merged_path):
            size_kb = os.path.getsize(merged_path) / 1024
            _log(f"Merged PDF saved: {merged_path}  ({size_kb:.1f} KB)", "OK")
        else:
            _log("Merge failed.", "ERROR")
        print()

    # ── summary ───────────────────────────────────────────────
    _log("=" * 58)
    _log("Batch conversion finished.")
    _log(f"  Converted : {len(success_paths) - skip_count}")
    _log(f"  Skipped   : {skip_count}")
    _log(f"  Failed    : {fail_count}")
    if MERGE_ALL_INTO_ONE_PDF:
        _log(f"  Merged PDF: {MERGED_PDF_FILENAME}")
    _log("=" * 58)


if __name__ == "__main__":
    main()